In [1193]:
import pandas as pd
from torch.utils.data import random_split, TensorDataset, DataLoader
import torch
import torch.nn as nn 
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_curve
import numpy as np

In [1194]:
selected_features = ['Pclass', 'Parch', 'Fare', 'Sex', 'Embarked', 'ticket_prefix']
target = 'Survived'

In [1195]:
train = pd.read_csv('prepared_train.csv')
train_X = train[selected_features]
train_y = train[target]
test = pd.read_csv('prepared_test.csv')
test_X = test[selected_features]

In [1196]:
train_X.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Pclass         891 non-null    int64  
 1   Parch          891 non-null    int64  
 2   Fare           891 non-null    float64
 3   Sex            891 non-null    str    
 4   Embarked       891 non-null    str    
 5   ticket_prefix  891 non-null    str    
dtypes: float64(1), int64(2), str(3)
memory usage: 41.9 KB


In [1197]:
train_X['ticket_prefix'] = train_X['ticket_prefix'].astype('category')
train_X['Embarked'] = train_X['Embarked'].astype('category')
train_X['Sex'] = train_X['Sex'].astype('category')

In [1198]:
test_X['ticket_prefix'] = test_X['ticket_prefix'].astype('category')
test_X['Embarked'] = test_X['Embarked'].astype('category')
test_X['Sex'] = test_X['Sex'].astype('category')

Использую One-Hot Encoding

In [1199]:
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), [name for name in selected_features if train_X[name].dtype != 'category']),
    ('cat', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), [name for name in selected_features if train_X[name].dtype == 'category'])
])

In [1200]:
X_test_processed = preprocessor.fit_transform(test_X)


print(f"Размерность после one-hot: {X_test_processed.shape[1]}")

X_test_tensor = torch.tensor(X_test_processed, dtype=torch.float32)

Размерность после one-hot: 21


In [1201]:
X_train_processed = preprocessor.transform(train_X)

print(f"Размерность после one-hot: {X_train_processed.shape[1]}")

X_train_tensor = torch.tensor(X_train_processed, dtype=torch.float32)
y_train_tensor = torch.tensor(train_y, dtype=torch.float32)

Размерность после one-hot: 21


In [1202]:
X_train_tensor[0]

tensor([ 0.8735, -0.4002, -0.5078,  0.0000,  1.0000,  0.0000,  0.0000,  1.0000,
         1.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000,  0.0000,  0.0000,  0.0000])

In [1203]:
train_size = int(0.8 * len(train_X))
val_size = len(train_X) - train_size

In [1204]:
train_dataset, val_dataset = random_split(TensorDataset(X_train_tensor, y_train_tensor), [train_size, val_size], generator=torch.Generator().manual_seed(42))

In [1205]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

Архитектура модели

In [1206]:
class BinaryClassification(nn.Module):
    def __init__(self, input_dim, hidden_dims=[128,64,32,16], drop_rate=0.3):
        super().__init__()
        
        layers = []
        prev_dim = input_dim

        for dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, dim),
                nn.BatchNorm1d(dim),
                nn.ReLU(),
                nn.Dropout(drop_rate)
            ])
            prev_dim = dim

        layers.append(nn.Linear(prev_dim, 1))

        self.net = nn.Sequential(*layers)

    def forward(self, X):
        return self.net(X)

In [1207]:
model = BinaryClassification(input_dim=X_train_processed.shape[1], hidden_dims=[128,64,32])

Сбалансируем классы

In [1208]:
all_labels = []
for _, y in train_loader:
    all_labels.extend(y.numpy())

from collections import Counter
print(Counter(all_labels))

neg, pos = Counter(all_labels)[0], Counter(all_labels)[1]
pos_weight = torch.tensor([neg / pos])

Counter({np.float32(0.0): 423, np.float32(1.0): 289})


Функция потерь

In [1209]:
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

Оптимизатор

In [1210]:
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0001, weight_decay=1e-3)

Планировщик для снижения lr

In [1211]:
scheduler = ReduceLROnPlateau(
    optimizer, 
    mode='min',
    factor=0.1, 
    patience=5
)


In [1212]:
for epoch in range(100):
    model.train()
    
    train_loss = 0.0
    for X, y in train_loader:
        optimizer.zero_grad()
        logits = model(X)
        loss = loss_fn(logits.squeeze(), y)
        train_loss += loss.item()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

    avg_train_loss = train_loss / len(train_loader)
    
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for X, y in val_loader:
            logits = model(X)
            
            preds = (logits.squeeze() > 0.5).float()
            correct += (preds == y).sum().item()
            total += y.size(0)
            
            loss = loss_fn(logits.squeeze(), y)
            val_loss += loss.item()
    
    avg_val_loss = val_loss / len(val_loader)
    val_accuracy = correct / total
    
    scheduler.step(avg_val_loss)
    
    if epoch % 10 == 0:
        print(f"Epoch: {epoch}")
        print(f"  Train Loss: {avg_train_loss:.4f}")
        print(f"  Val Loss: {avg_val_loss:.4f}")
        print(f"  Val Accuracy: {val_accuracy:.4f}")

Epoch: 0
  Train Loss: 0.8588
  Val Loss: 0.8081
  Val Accuracy: 0.7039
Epoch: 10
  Train Loss: 0.6809
  Val Loss: 0.6410
  Val Accuracy: 0.8045
Epoch: 20
  Train Loss: 0.6153
  Val Loss: 0.5710
  Val Accuracy: 0.8380
Epoch: 30
  Train Loss: 0.5717
  Val Loss: 0.5501
  Val Accuracy: 0.8324
Epoch: 40
  Train Loss: 0.6064
  Val Loss: 0.5278
  Val Accuracy: 0.8268
Epoch: 50
  Train Loss: 0.5393
  Val Loss: 0.5244
  Val Accuracy: 0.8324
Epoch: 60
  Train Loss: 0.5709
  Val Loss: 0.5236
  Val Accuracy: 0.8380
Epoch: 70
  Train Loss: 0.5640
  Val Loss: 0.5163
  Val Accuracy: 0.8380
Epoch: 80
  Train Loss: 0.5667
  Val Loss: 0.5162
  Val Accuracy: 0.8324
Epoch: 90
  Train Loss: 0.5613
  Val Loss: 0.5196
  Val Accuracy: 0.8380


In [1213]:
model.eval()

all_logits = []
all_labels = []
optimal_threshold = 0.0
with torch.no_grad():
    for X, y in val_loader:
        logits = model(X)
        all_logits.extend(logits.squeeze().numpy())
        all_labels.extend(y.numpy())
    
    fpr, tpr, thresholds = roc_curve(all_labels, all_logits)
    optimal_idx = np.argmax(tpr - fpr)
    optimal_threshold = thresholds[optimal_idx]

val_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for X, y in val_loader:
        logits = model(X)
        
        preds = (logits.squeeze() > optimal_threshold).float()
        correct += (preds == y).sum().item()
        total += y.size(0)
        
        loss = loss_fn(logits.squeeze(), y)
        val_loss += loss.item()

avg_val_loss = val_loss / len(val_loader)
val_accuracy = correct / total

print(val_accuracy)

0.8100558659217877


In [1214]:
ans = []
with torch.no_grad():
    logits = model(X_test_tensor)    
    ans = (logits.squeeze() > optimal_threshold).float()

In [1215]:
ans

tensor([0., 1., 0., 0., 1., 0., 1., 1., 1., 0., 0., 0., 1., 0., 1., 1., 0., 0.,
        0., 1., 0., 0., 1., 0., 1., 0., 1., 0., 0., 0., 0., 0., 0., 0., 1., 0.,
        1., 1., 0., 0., 0., 0., 0., 1., 1., 0., 0., 0., 1., 0., 1., 0., 1., 1.,
        0., 0., 0., 0., 0., 1., 0., 0., 0., 1., 1., 1., 1., 1., 1., 1., 1., 0.,
        1., 0., 1., 1., 0., 1., 0., 1., 0., 1., 0., 0., 0., 0., 1., 0., 1., 1.,
        1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 0., 0., 1., 0., 0., 0.,
        0., 0., 0., 1., 1., 1., 1., 0., 0., 1., 1., 1., 1., 0., 1., 0., 0., 1.,
        0., 1., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 0.,
        0., 0., 1., 0., 0., 1., 1., 0., 0., 1., 0., 0., 1., 1., 0., 0., 1., 0.,
        1., 0., 0., 1., 0., 0., 1., 1., 0., 0., 0., 0., 0., 1., 1., 1., 1., 1.,
        0., 0., 1., 0., 1., 0., 1., 0., 0., 0., 0., 0., 0., 0., 1., 0., 1., 1.,
        0., 1., 1., 0., 1., 1., 0., 1., 1., 0., 1., 0., 0., 0., 0., 1., 1., 1.,
        1., 1., 1., 0., 1., 0., 1., 0., 

In [1220]:
submission = pd.DataFrame()
submission['PassengerId'] = test['PassengerId']
submission['Survived'] = ans
submission['Survived'] = submission['Survived'].astype('int')

In [1221]:
submission.to_csv('submission.csv', index=False)